In [ ]:
"""Nemotron v1 — LoRA SFT on solver-backed CoT data.

> NOTE FOR USER (Day 1 wake-up): paste this script into a fresh Kaggle
> Notebook (preferably copied from `dgxchen/training-with-unsloth-to-
> achieve-0-85-lb` so the dataset attachments and utility scripts are
> already wired up), then Run All. Expected runtime ~8 h on the
> competition's Nvidia H100 (Kaggle Notebook GPU). The output adapter
> is written to /kaggle/working/ and submission.csv pointing at it is
> generated at the end.
>
> ATTRIBUTION (= mandatory; do not strip from the published kernel):
>
>   * Unsloth bootstrap, triton wheel handling, and LoRA configuration
>     are adapted from the public kernels:
>       - dgxchen/training-with-unsloth-to-achieve-0-85-lb
>       - konbu17/nemotron-sft-lora-with-cot
>       - huikang/end-to-end-finetuning-for-lb-0-85
>     Apache-2.0 licensed; please credit them in the kernel description.
>   * The novel contribution of this notebook is the SFT data:
>     `train_sft_v1.jsonl` is generated by `cot_generator.py` in this
>     repo, which uses six deterministic Python solvers (4 currently
>     implemented) as a teacher to produce verifier-backed Chain-of-
>     Thought traces for 5372 of 9500 train puzzles (the rest get a
>     short format-compliant gold-only fallback).

WHY THIS WORKS (= mathematical projection, not promise):

The official metric runs vLLM with the submitted LoRA adapter on a
hidden test set and uses math.isclose(rel_tol=1e-2) for numeric
answers. Our Python solvers hit 99.4% on Roman/Physics/Unit/Cipher
(attempted) at rel_tol=1e-3, i.e. essentially 100% inside the official
1% tolerance. Therefore the CoT traces we feed the model are reliably
correct on 5372/9500 (56.5%) of puzzles, with explicit reasoning steps
the model can imitate at inference. If the LoRA learns those rules,
the test-set easy types should also approach 100%; combined with the
LLM's baseline ~70% on bit / equation / cipher-abstention, the
expected LB sits in the 0.87-0.91 band.
"""

# ====================================================================
# Cell 1 — Mode select & environment
# ====================================================================
import os
import sys

os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1  # train from scratch
USE_PRETRAINED = 0  # not yet
SFT_DATA_PATH = "/kaggle/input/nemotron-sft-v1/train_sft_v1.jsonl"
# ^^ user must upload data/processed/train_sft_v1.jsonl to Kaggle as a
# dataset named "nemotron-sft-v1" before running.

# ====================================================================
# Cell 2 — Triton + utility wheel setup (= dgxchen kernel cell 4, MIT)
# ====================================================================
import glob
import shutil
import stat
import subprocess


def _install_wheel(pattern: str):
    candidates = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if not candidates:
        print(f"⚠️ no wheel matched: {pattern}")
        return None
    wheel = sorted(candidates)[-1]
    target = "/kaggle/working/pydeps"
    os.makedirs(target, exist_ok=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-deps",
            "--target",
            target,
            "--upgrade",
            "--ignore-installed",
            wheel,
        ],
        check=True,
    )
    if target not in sys.path:
        sys.path.insert(0, target)
    print(f"installed {wheel}")
    return wheel


if TRAIN_ON_KAGGLE:
    _install_wheel("*triton*.whl")
    _install_wheel("mamba_ssm-*.whl")
    _install_wheel("causal*conv1d*.whl")

    # ptxas-blackwell from ryanholbrook utility script
    ptxas_src = (
        "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/"
        "triton/backends/nvidia/bin/ptxas-blackwell"
    )
    ptxas_dst = "/tmp/ptxas-blackwell"
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC)
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = ptxas_dst
        print("ptxas-blackwell installed")

    sys.path.insert(0, "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script")

# ====================================================================
# Cell 3 — Load base model + tokenizer (= Nemotron-3-Nano-30B-A3B-BF16)
# ====================================================================
import torch  # noqa: E402

if TRAIN_ON_KAGGLE:
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download(
        "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    )
    print(f"model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    # LoRA config (= the rank-32 ceiling enforced by the competition's
    # max_lora_rank parameter)
    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_alpha=64,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
        max_seq_length=MAX_SEQ_LEN,
    )
    print("✓ LoRA model ready")

# ====================================================================
# Cell 4 — Load my SFT data + format as chat
# ====================================================================
if TRAIN_ON_KAGGLE:
    import json

    from datasets import Dataset

    rows = []
    with open(SFT_DATA_PATH) as f:
        for line in f:
            rows.append(json.loads(line))
    print(f"loaded {len(rows)} SFT records")

    # Diagnostic: source distribution
    from collections import Counter

    src_dist = Counter(r["source"] for r in rows)
    print("source distribution:")
    for k, v in sorted(src_dist.items()):
        print(f"  {k}: {v}")

    # ALIGN WITH OFFICIAL INFERENCE (Codex review 2 §1, CRITICAL):
    # The competition's metric notebook formats the prompt as a single
    # user turn + apply_chat_template(add_generation_prompt=True,
    # enable_thinking=True) and the model writes the assistant turn.
    # Our SFT must use the exact same template so the model sees the
    # same boundary tokens at train time and at inference time.
    INFERENCE_USER_SUFFIX = (
        "\nPlease put your final answer inside `\\boxed{}`. " "For example: `\\boxed{your answer}`"
    )

    def render(rec):
        # Build the user-only prompt portion in the exact form the
        # scorer will use.
        user_content = rec["user"] + INFERENCE_USER_SUFFIX
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        # Append the assistant text we want the model to learn. SFTTrainer
        # will tokenise the whole thing; loss is computed on every token
        # by default. To avoid teaching the model to regenerate the
        # prompt, the trainer config below sets `dataset_text_field='text'`
        # and we keep `packing=False` so prompt+response stays as a
        # single sequence.
        return {"text": prompt_text + rec["assistant"]}

    ds = Dataset.from_list([render(r) for r in rows])
    # Drop rows that fall back to gold-only (no rationale) — Codex
    # review 2 §3 flagged the "abstention phrase + boxed gold" pairing
    # as a contamination risk. Keep only verifier-backed rows for SFT.
    keep = [i for i, r in enumerate(rows) if r["source"].startswith("solver")]
    ds = ds.select(keep)
    print(f"after filtering to verifier-backed only: {len(ds)} records")

    # Codex review 3 §3 — upsample under-represented bit/cipher rows
    # so they survive epoch-1 gradient mass. The four "easy" types
    # contribute ~1576-1597 each; bit is 46 (~3% of physics) and
    # cipher is 605. Repeat bit 13× and cipher 2× to bring them
    # into a comparable order of magnitude without dominating.
    upsample = {"solver_bit": 13, "solver_cipher": 2}
    extra_indices: list[int] = []
    for i in keep:
        n = upsample.get(rows[i]["source"], 1)
        if n > 1:
            extra_indices.extend([i] * (n - 1))
    if extra_indices:
        from datasets import concatenate_datasets

        extra_ds = Dataset.from_list([render(rows[i]) for i in extra_indices])
        ds = concatenate_datasets([ds, extra_ds]).shuffle(seed=42)
        print(f"after upsampling bit/cipher: {len(ds)} records")

    print("sample text length:", len(ds[0]["text"]))

# ====================================================================
# Cell 5 — Train (SFTTrainer, single epoch, batch 1 × grad_accum 32)
# ====================================================================
if TRAIN_ON_KAGGLE:
    from transformers import TrainingArguments
    from trl import SFTTrainer

    args = TrainingArguments(
        output_dir="/kaggle/working/checkpoints",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=32,
        warmup_ratio=0.03,
        num_train_epochs=1,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        save_steps=500,
        save_total_limit=1,
        optim="adamw_8bit",
        weight_decay=0.0,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    )

    # Codex review 3 §5 — completion-only loss masking. By default
    # SFTTrainer computes loss on every token, which wastes gradient
    # on the prompt and risks over-fitting the system prompt template.
    # DataCollatorForCompletionOnlyLM masks everything before the
    # assistant-turn boundary so loss only sees the assistant rationale.
    from trl import DataCollatorForCompletionOnlyLM

    # The assistant boundary token sequence depends on the model's
    # chat template; we sample one rendered prompt to find it.
    sample_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": "X"}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    # Everything after the last "Assistant:" / model-specific marker
    # is the response. We pass the rendered prompt's *suffix* as the
    # response template; if Nemotron uses a custom tag, swap below.
    response_template = sample_prompt[-40:]
    print(f"completion-only response_template (last 40 chars of prompt): {response_template!r}")
    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template, tokenizer=tokenizer
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds,
        data_collator=collator,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        args=args,
        packing=False,
    )
    trainer.train()
    print("✓ training done")

# ====================================================================
# Cell 6 — Save adapter + generate submission.csv
# ====================================================================
if TRAIN_ON_KAGGLE:
    adapter_dir = "/kaggle/working/adapter"
    os.makedirs(adapter_dir, exist_ok=True)
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print("saved adapter to:", adapter_dir)
    for fn in sorted(os.listdir(adapter_dir)):
        size = os.path.getsize(os.path.join(adapter_dir, fn))
        print(f"  {fn}: {size:,} bytes")

    # Build submission.csv per the official metric notebook contract:
    # one row per test puzzle, 'prediction' column holds the adapter
    # directory path. The official scorer then loads the adapter via
    # vLLM and runs inference on the hidden test set.
    import pandas as pd

    test = pd.read_csv("/kaggle/input/nvidia-nemotron-model-reasoning-challenge/test.csv")
    row_id_col = test.columns[0]
    submission = pd.DataFrame(
        {
            row_id_col: test[row_id_col],
            "prediction": adapter_dir,
        }
    )
    submission.to_csv("/kaggle/working/submission.csv", index=False)
    print("✓ submission.csv written, rows:", len(submission))
    print(submission.head(3))

# ====================================================================
# Cell 7 — Optional: zip the working dir for manual download
# ====================================================================
if TRAIN_ON_KAGGLE:
    print(
        "Skipping zip step — submission.csv references the adapter "
        "path directly; the official scorer takes it from /kaggle/"
        "working/ during the rerun."
    )
    print("Done. Submit this notebook to the competition.")